# (method) BASIC CRYSTAL ENSEMBLE WORK FLOWS

This notebook guides you through the various approaches that exist with regards to crystal structures using the ASE SDK. Generated crystal structures have the ability to be introduced to defects through a considerably simple method -- by simply removing individual atosm using the del command to remove specified ato indexes from a ASE.cell. The generated Atoms represnetation of the final imperfect supercell is then subjected to a PDB crreation workflow, which results in a the corresponding .pdb file for the generated crystal strucutre. 

A correctly formated .pdb file is later subject to the ensamble parser which calcualtes the energy ocer the whole system, enabling approximative calculations of the lattice and defect energies of individual crystals. 

> Note: the crystals made here are incredibly basic, polymorphism and proton-disorder are but a handful of phenomena that complexify real crystal prediction and related modeling.

## Constructing a unit cell of the Atoms object using the ASE package manually

Import necessary packages

In [ ]:
import numpy as np
import ase.build.bulk as bulk
from ase import Atoms
from ase.build import molecule
from ase.io import write

Define lattice parameter of a cubed cell, and fcc fractional coordinates

In [ ]:
# 1. Define the lattice parameter for your FCC cell (in Angstroms)
a = 4.5 

# 2. Define the fractional coordinates of an FCC lattice
fcc_fractional_coords = [
    (0.0, 0.0, 0.0),
    (0.5, 0.5, 0.0),
    (0.5, 0.0, 0.5),
    (0.0, 0.5, 0.5)
]

Initalize a cubic Atoms by defining the cell property using the singular lattice parameter. 

In [ ]:
# 3. Initialize an empty Atoms object with a cubic cell
unit_cell = Atoms(cell=[a, a, a], pbc=True)


Place water molecules across the fractional fcc coordinates

In [ ]:

# 4. Get a standard water molecule from ASE's database
base = molecule('H2O')

# 5. Place a water molecule at each FCC lattice point
for frac_coord in fcc_fractional_coords:
    # Convert fractional coordinates to Cartesian (x, y, z)
    cartesian_pos = np.array(frac_coord) * a
    
    # Create a copy of the base water molecule
    mol = base.copy()
    
    # Move the oxygen atom of the molecule to the lattice point
    mol.translate(cartesian_pos)
    
    # Add the molecule to our unit cell
    unit_cell.extend(mol)

Create a supercell from the newly constructed unit cell and write to .pdb

In [ ]:
# 6. Create a 2x1x1 supercell to get exactly 8 molecules (4 molecules * 2)
# If you wanted a standard 2x2x2 supercell, it would be unit_cell.repeat((2, 2, 2))
eight_molecule_system = unit_cell.repeat((2, 1, 1))

# 7. Write the output to a PDB file
write('water_fcc_8_molecules.pdb', eight_molecule_system)

print("PDB file successfully generated!")

## Constructing bulk structures with ASE

This particular workflow works only with g2 structures.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ase.build import bulk
from ase.calculators.emt import EMT
from ase.eos import EquationOfState
from ase.filters import FrechetCellFilter
from ase.optimize import BFGS
from ase.units import kJ
from ase.visualize.plot import plot_atoms

Generate and plot a bulk Atoms object.

In [ ]:
atoms = bulk('Ag')

fig, ax = plt.subplots()
plot_atoms(atoms * (3, 3, 3), ax=ax)
ax.set_xlabel(r'$x (\AA)$')
ax.set_ylabel(r'$y (\AA)$')
fig.tight_layout()

## Buidling crystals

Note that we only aim to produce the corresponding PDB file of a supercell with ONE defect here. MORE defects could be introduced later on.

In [ ]:
import matplotlib.pyplot as plt

from ase.build import bulk
from ase.geometry import get_distances
from ase.visualize.plot import plot_atoms

#### Bulk lattice configurations from build.bulk

In [ ]:
def add_decoration(ax, conf, centralidx, neighborcutoff, arrow_offset=(0, 0)):
    # function for adding arrows to neighboring atoms
    vectors, distances = get_distances(
        conf.positions[centralidx], p2=conf.positions
    )
    neighboridx = [
        i for i, j in enumerate(distances[0]) if (neighborcutoff >= j)
    ]

    # we normalize the size of the arrows for illustration purposes
    norm = neighborcutoff / 5
    for idx in neighboridx:
        ax.arrow(
            conf.positions[centralidx][0] + arrow_offset[0],
            conf.positions[centralidx][1] + arrow_offset[1],
            vectors[0][idx][0] / norm,
            vectors[0][idx][1] / norm,
            width=0.1,
            color='k',
        )

Lattice parameters

In [ ]:
lattice_constant = 8
centralidx = 4
neighborcutoff = 8

Square Lattice

In [ ]:
# square lattice
# build structure
conf = bulk('Po', a=lattice_constant)
# moving the atom in the middle of the cell
positions = conf.get_positions()
positions += conf.cell[0][0] / 2
conf.set_positions(positions)
conf = conf.repeat((3, 3, 1))

# plot cs structure
fig, ax = plt.subplots()
add_decoration(ax, conf, centralidx, neighborcutoff, arrow_offset=(0.5, 0.5))
plot_atoms(conf, ax, offset=(0, 0))  # , rotation=('-80x,0y,0z'))
ax.set_axis_off()
ax.text(
    0.1,
    -0.1,
    'square lattice: r$_1$=a, Z$_1$=4',
    transform=ax.transAxes,
    fontsize=16,
)
plt.show()

Rectangular lattice

In [ ]:
# rectangular lattice
# build structure
conf = bulk(
    'C',
    crystalstructure='orthorhombic',
    a=lattice_constant,
    b=lattice_constant / 2,
    c=lattice_constant / 2,
)

# moving the atom in the middle of the cell
positions = conf.get_positions()
positions += [conf.cell[0][0] / 2, conf.cell[0][0] / 4, conf.cell[0][0] / 4]
conf.set_positions(positions)
conf = conf.repeat((3, 3, 1))

# plot orc structure
fig, ax = plt.subplots()
add_decoration(ax, conf, centralidx, neighborcutoff, arrow_offset=(0.5, 0.5))
plot_atoms(conf, ax, offset=(0, 0))
ax.set_axis_off()
ax.text(
    0,
    -0.2,
    'rectangular lattice with a 2:1 aspect ratio:\n r$_1$=a/2, Z$_1$=2',
    transform=ax.transAxes,
    fontsize=16,
)
plt.show()

Hexagonal lattice

In [ ]:
# hexagonal lattice
# build structure
conf = bulk('Be', a=lattice_constant)
# here, we slice the cell to have one a 2D layer of atoms
confmask = [i.index for i in conf if i.position[2] < 1]
conf = conf[confmask]
conf = conf.repeat((3, 3, 1))
positions = conf.get_positions()
positions -= conf.positions[0]
conf.set_positions(positions)

# plot hpc structure
fig, ax = plt.subplots()
add_decoration(
    ax,
    conf,
    centralidx,
    neighborcutoff,
    arrow_offset=(-conf.cell[1][0] + 1, 1.5),
)
plot_atoms(conf, ax, offset=(0, 0), rotation=('0x,0y,0z'))
ax.set_axis_off()
ax.text(
    0.1,
    -0.1,
    'hexagonal lattice: r$_1$=1.075a, Z$_1$=6',
    transform=ax.transAxes,
    fontsize=16,
)
plt.show()

#### Crystal structures from spacegroup.crystal module and Atoms.repeat

> ase.spacegroup.crystal 

Creates a conventional unit cell for specified common crystals

> ase.Atoms.repeat

Repeats a porivded cell of the Atoms object. Does not guarantee formation of supercell. This is instead performed uing the ase.build.make_supercell which creates a supercell object.

In [ ]:
from ase.spacegroup import crystal
from ase.build import find_optimal_cell_shape


In [ ]:
a = 4.5
unit_cell = Atoms(cell=[a, a, a])
cell = unit_cell.cell

ice = crystal(                          # Atoms object. 
    symbols='H2O',
    basis=[(0, 0, 0), (0.5, 0.5, 0.5)], # Relative positions in the atoms
    spacegroup=225,
    cellpar=[a,a,a]                     # Defines cell parameters. 
)

isinstance(ice, ase.Atoms)

print(ice)
print(len(ice))
print(
    ice[0],"\n", 
    ice[1],"\n",
    ice[2],"\n",
    ice[3],"\n",
    ice[4], "\n",
    ice[5], "\n",
    ice[6], "\n",
    ice[7], "\n",
)
plot_atoms(ice, rotation='60x,60y,50z')
plot_atoms(ice, rotation='60x,60y,60z')
plot_atoms(ice, rotation='60x,60y,70z')




In [ ]:
ice_block = ice.repeat((4, 2, 2))

# isinstance(ice_block, ase.Atoms)

# P = find_optimal_cell_shape(
#     cell=ice_block.cell,
#     target_size=100,
#     target_shape='sc'
#     )
# ice_block = ase.build.supercells

print(ice_block)
print(len(ice_block))
print(
    ice_block[0],"\n", 
    ice_block[1],"\n",
    ice_block[2],"\n",
    ice_block[3],"\n",
    ice_block[4], "\n",
    ice_block[5], "\n",
    ice_block[6], "\n",
    ice_block[7], "\n",
)
plot_atoms(ice_block, rotation='60x,60y,50z')
plot_atoms(ice_block, rotation='60x,60y,60z')
plot_atoms(ice_block, rotation='60x,60y,70z')

#### Crystal structure using build.find_optimal_cell_shape and buidl.make_supercells

In [ ]:
a = 4.5
unit_cell = Atoms(cell=[a, a, a])
cell = unit_cell.cell

ice = crystal(                          # Atoms object. 
    symbols='H2O',
    basis=[(0, 0, 0), (0.5, 0.5, 0.5)], # Relative positions in the atoms
    spacegroup=225,
    cellpar=[a,a,a]                     # Defines cell parameters. 
)

isinstance(ice, ase.Atoms)

print(ice)
print(len(ice))
print(
    ice[0],"\n", 
    ice[1],"\n",
    ice[2],"\n",
    ice[3],"\n",
    ice[4], "\n",
    ice[5], "\n",
    ice[6], "\n",
    ice[7], "\n",
)
plot_atoms(ice, rotation='60x,60y,50z')
plot_atoms(ice, rotation='60x,60y,60z')
plot_atoms(ice, rotation='60x,60y,70z')

Calculate the transformation matrix for the transformation from unit cell to supercell at a specified size.

In [ ]:
P = find_optimal_cell_shape(
    cell=ice.cell,
    target_size=100,
    target_shape='sc'
    )

Generate the supercell using the cell object and transformation matrix

In [ ]:
ice_block = make_supercell(ice, P)
#fig, ax = plt.subplots()
plot_atoms(ice_block, rotation='40x,60y,50z')
plot_atoms(ice_block, rotation='50x,60y,50z')
plot_atoms(ice_block, rotation='60x,60y,50z')
#ax.set_axis_off()

## Introducing defects into supercells

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ase.build import find_optimal_cell_shape
from ase.build import make_supercell
from ase.build.supercells import eval_length_deviation
from ase.build import bulk
from ase.visualize.plot import plot_atoms

Create a bulk Atoms object.

In [ ]:
conf = bulk('Au') # Bulid() returns an atom object with the primitive cell of the specified crystal. 

Calculate the transformation matrix for supercell of specific size.

In [ ]:
P = find_optimal_cell_shape(conf.cell, 495, 'sc')
print(P)

P1 = find_optimal_cell_shape(conf.cell, 32, 'sc')
print(P1)

P2 = find_optimal_cell_shape(conf.cell, 495, 'sc')
print(P2)

Generating the supercell.

In [ ]:
supercell = make_supercell(conf, P)

fig, ax = plt.subplots()
plot_atoms(supercell, ax)
ax.set_axis_off()

Introduce a defect by removing specific atoms

In [ ]:
del supercell[300:494]

fig, ax = plt.subplots()
plot_atoms(supercell, ax)
ax.set_axis_off()

## Writing and Formating Atoms obj. to .pdb (Not finished)

#### Write Atoms to file using ase.io.write

In [ ]:
from ase.io import write
from ase.spacegroup import crystal
from ase.build import find_optimal_cell_shape
from ase.build import make_supercell
from ase.visualize.plot import plot_atoms
import matplotlib.pyplot as plt

Create an O block

In [ ]:
ice = crystal('O', basis=[(1/3, 2/3, 0.0625)], spacegroup=194, cellpar=[4.52, 4.52, 7.37, 90, 90, 120])
P = find_optimal_cell_shape(ice.cell, target_size=100, target_shape='sc')
ice = make_supercell(ice, P)
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
plot_atoms(ice, ax=ax[0], rotation='60x,60y,50z')
plot_atoms(ice, ax=ax[1], rotation='60x,60y,130z')
plt.show()

#### Specify custom lables in the PDB file 
\# TODO: Finish up the PDB creation pipeline, and optimize the ensamble parser implementation and related auzilirary workflows if there are any.

In [ ]:
# Extract text from pdb file, 
from Bio.PDB import PDBParser, PDBIO

In [ ]:
#TODO: Implement file handling to format the pdb file to be compatible with vlx.EnsambleParser.

ep = EnsembleParser() # Initate an ensamble parser instance.
print("Ensemble parser instance created.")
ensemble = ep.structures(
    trajectory_file = "water_fcc_8_molecules.pdb", # One snap shot pdb. 
    num_snapshots = None, 
    qm_region = "resname LIG", 
    pe_cutoff = 6.0, 
    # npe_cutoff = 6.01
)
print(ensemble)
print("Ensemble parser instance populated.")
ed = EnsembleDriver() # Initate an ensamble driver instance.
print("Ensemble driver instance created.")
ed.set_env_models(pe_model = 'SEP', npe_model = 'tip3p')
print("Ensemble driver instance populated.")

ed.compute(ensemble, basis_set = '6-31G')

## Generating Crystal Structures with Higher Complexity

### Generating ice crystals

#### Proton-Disorder and topology generation
A major problem in ice crystal modeling is the disordered arrangmenet of protons that maniferst around ordered lattices of oxygen -- with oxygen making up the spacegroup back bone of any ice crystal. 

In order to generate appropriate estimations for ice crystals the related topology linking ocygens to protons in a grpah like manner needs to be deduced. 

> **GenIce:** A Swiss army knife to generate hydrogen-disordered ice structures. The program generates various ice lattice with proton disorder and without defect. Total dipole moment is always set to zero (except the case you specify --nodep option). The minimal structure (with --rep 1 1 1 option) is not always the unit cell of the lattice because it is difficult to deal with the hydrogen bond network topology of tiny lattice under periodic boundary condition. Note that the generated structure is not optimal according to the potential energy.

Read more about GenIce here: https://pypi.org/project/GenIce/

Read more about ice Ih here: https://www.researchgate.net/figure/Left-Two-views-of-the-crystal-structure-of-ice-Ih-showing-a-lattice-of-puckered_fig4_30759481

Generate a propper Ice 1h with the command <genice2 --rep 8 8 8 1h --format xyz > 1hx888.xyz> 

In [ ]:
import os

In [ ]:
path = '../../.crystal_structs'
os.system(f'genice2 --rep 1 1 1 1h --format cif > {path}/1hx111.cif')
os.system(f'ase convert {path}/1hx111.cif {path}/1hx111.pdb')

Now read the files.

In [ ]:
from ase.io import read, iread, write
from ase.visualize.plot import plot_atoms
from ase.build import find_optimal_cell_shape
from ase.build import make_supercell

In [ ]:
path = '../../.crystal_structs'
filename = '1hx111.pdb'
ice = read(f'{path}/{filename}')

plot_atoms(ice, rotation='60x,60y,50z')


Create supercell

In [ ]:
P = find_optimal_cell_shape(
    cell=ice.cell,
    target_size=10,
    target_shape='sc'
    )   
ice_block = make_supercell(ice, P)
write(f'{path}/1hx111_supercell.cif', ice_block)
plot_atoms(ice_block, rotation='60x,60y,50z')

In [ ]:
plot_atoms(ice_block, rotation='60x,60y,50z')
plot_atoms(ice_block, rotation='60x,60y,70z')
plot_atoms(ice_block, rotation='60x,60y,90z')

Convert to pdb

In [ ]:
write(f'{path}/1hx111_supercell.pdb', ice_block)

Calculating energies

In [ ]:
import veloxchem.ensembleparser
import veloxchem.ensembledriver

In [ ]:
ep = veloxchem.ensembleparser.EnsembleParser()   
ed = veloxchem.ensembledriver.EnsembleDriver()

print("1. \t Ensemble parser instance created.")
ensemble = ep.structures(
    trajectory_file = f"{path}/1hx111.pdb", # One snap shot pdb. 
    num_snapshots = None, 
    qm_region = "resname LIG", 
    pe_cutoff = None
)

print(f'2. \t Ensemble Configuration:\n\t==================== \n\t{ensemble} \n\t ====================\n')

ed.set_env_models(pe_model = 'SEP', npe_model = 'tip3p')

ed.compute(ensemble, basis_set = '6-31G')

Introducting a defect into it

In [ ]:
ice_block